# AC-1 — AI-Generated Image Detection

Detect whether an image was created by generative AI (Midjourney, Stable Diffusion, DALL·E, etc.) or manipulated with image editing tools.

**Model:** `AC-1`  
**Input:** Image files (`.jpg`, `.jpeg`, `.png`, etc.)

---
### Contents
1. Setup
2. One-call detection
3. Two-step: upload now, poll later
4. Visualize results — heatmap
5. Error handling
6. Async client

---
## 1. Setup

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('../src'))

from authenta.authenta_client import AuthentaClient
from authenta import (
    AuthentaError,
    AuthenticationError,
    QuotaExceededError,
    InsufficientCreditsError,
)

CLIENT_ID     = "YOUR_CLIENT_ID"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"
BASE_URL      = "https://platform.authenta.ai"

client = AuthentaClient(
    base_url=BASE_URL,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

print("Client ready.")

---
## 2. One-call Detection

`client.process()` uploads the file and blocks until the result is ready.

In [ ]:
IMAGE = "../data_samples/nano_img2.png"

media = client.process(IMAGE, model_type="AC-1")

print(f"Media ID   : {media['mid']}")
print(f"Status     : {media['status']}")
print(f"Is Fake    : {media.get('fake')}")
print(f"Result URL : {media.get('resultURL')}")
print(f"Heatmap    : {media.get('heatmapURL')}")

---
## 3. Two-step: Upload Now, Poll Later

Use `upload_file()` to start the job, then call `wait_for_media()` when you're ready to collect the result.

In [ ]:
# Step 1 — upload
upload_meta = client.upload_file(IMAGE, model_type="AC-1")
mid = upload_meta["mid"]
print(f"Uploaded.  Media ID : {mid}")
print(f"           Status   : {upload_meta['status']}")

In [ ]:
# Step 2 — poll until done
media = client.wait_for_media(mid, interval=5.0, timeout=300.0)
print(f"Status  : {media['status']}")
print(f"Is Fake : {media.get('fake')}")

---
## 4. Visualize Results — Heatmap

Generate a heatmap overlay showing which regions of the image are flagged as manipulated.

In [ ]:
from authenta.visualization import save_heatmap

media = client.process(IMAGE, model_type="AC-1")

os.makedirs("results", exist_ok=True)
save_heatmap(
    media=media,
    out_path="results/ac1_heatmap.jpg",
    model_type="AC-1",
)
print("Saved: results/ac1_heatmap.jpg")

In [ ]:
from IPython.display import Image as IPImage
IPImage("results/ac1_heatmap.jpg", width=600)

---
## 5. Error Handling

In [ ]:
try:
    media = client.process(IMAGE, model_type="AC-1")
    print(f"Is Fake: {media.get('fake')}")
except AuthenticationError:
    print("Authentication failed — check your CLIENT_ID and CLIENT_SECRET.")
except QuotaExceededError:
    print("API quota exceeded — upgrade your plan.")
except InsufficientCreditsError:
    print("Not enough credits.")
except TimeoutError as e:
    print(f"Timed out: {e}")
except AuthentaError as e:
    print(f"API error [{e.code}]: {e.message}")

---
## 6. Async Client

In [ ]:
import asyncio
from authenta.async_authenta_client import AsyncAuthentaClient

# One-call — async
async def detect_single():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        media = await client.process(IMAGE, model_type="AC-1")
        print(f"Status  : {media['status']}")
        print(f"Is Fake : {media.get('fake')}")

await detect_single()

In [ ]:
# Two-step — async
async def detect_two_step():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        upload_meta = await client.upload_file(IMAGE, model_type="AC-1")
        mid = upload_meta["mid"]
        print(f"Uploaded: {mid}")

        media = await client.wait_for_media(mid)
        print(f"Status  : {media['status']}")
        print(f"Is Fake : {media.get('fake')}")

await detect_two_step()